In [51]:
import pandas as pd
import numpy as np

In [73]:
df = pd.read_csv("data/raw/new_airbnb_transactions.csv")

In [74]:
print(df.columns)

Index(['date', 'arriving_by_date', 'type', 'booking_date', 'start_date',
       'end_date', 'nights', 'guest', 'listing', 'details', 'gross_earnings',
       'earnings_year', 'lead_time', 'month', 'weekday', 'weekday_dept',
       'is_weekend', 'price_per_night'],
      dtype='object')


In [75]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 581 entries, 0 to 580
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              581 non-null    object 
 1   arriving_by_date  0 non-null      float64
 2   type              581 non-null    object 
 3   booking_date      581 non-null    object 
 4   start_date        581 non-null    object 
 5   end_date          581 non-null    object 
 6   nights            581 non-null    float64
 7   guest             581 non-null    int64  
 8   listing           581 non-null    object 
 9   details           0 non-null      float64
 10  gross_earnings    581 non-null    float64
 11  earnings_year     581 non-null    float64
 12  lead_time         581 non-null    int64  
 13  month             581 non-null    int64  
 14  weekday           581 non-null    object 
 15  weekday_dept      581 non-null    object 
 16  is_weekend        581 non-null    bool   
 1

,arriving_by_date,nights,guest,details,gross_earnings,earnings_year,lead_time,month,price_per_night
count,0.0,581.000000,581.000000,0.0,581.000000,581.000000,581.000000,581.000000,581.000000
mean,NaN,2.099828,273.820998,NaN,3370.929432,2023.309811,5.753873,6.743546,1513.536882
std,NaN,1.821429,157.779131,NaN,3616.944520,1.165772,13.600190,3.932357,507.253457
min,NaN,1.000000,1.000000,NaN,800.000000,2022.000000,0.000000,1.000000,250.000000
25%,NaN,1.000000,135.000000,NaN,1300.000000,2022.000000,0.000000,3.000000,1200.000000
50%,NaN,1.000000,275.000000,NaN,1800.000000,2023.000000,1.000000,7.000000,1500.000000
75%,NaN,3.000000,409.000000,NaN,4000.000000,2024.000000,4.000000,11.000000,1620.000000
max,NaN,14.000000,549.000000,NaN,33200.000000,2026.000000,149.000000,12.000000,4000.000000


In [58]:
#standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns)

Index(['date', 'arriving_by_date', 'type', 'booking_date', 'start_date',
       'end_date', 'nights', 'guest', 'listing', 'details', 'gross_earnings',
       'earnings_year', 'lead_time', 'month', 'weekday', 'weekday_dept',
       'is_weekend', 'price_per_night'],
      dtype='object')


In [59]:
#convert dates column to datetime
df["booking_date"] = pd.to_datetime(df["booking_date"])
df["start_date"] = pd.to_datetime(df["start_date"])
df["end_date"] = pd.to_datetime(df["end_date"])

In [60]:
#filter the column "Type" to keep only reservation ( removes extra tax rows)
df = df[df["type"] == "Reservation"]

In [64]:
#convert string to numeric columns
numeric_cols = ["nights" , "gross_earnings"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [66]:
#remove invalid rows
df = df.dropna(subset=["start_date", "nights", "gross_earnings"])

In [67]:
#create new features
#lead time- how many days before arrival the guest booked
df["lead_time"] = (df["start_date"] - df["booking_date"]).dt.days

# Extracting month feature
df["month"] = df["start_date"].dt.month

# extracting weekday arrival and departure feature
df["weekday"] = df["start_date"].dt.day_name()
df["weekday_dept"] = df["end_date"].dt.day_name()

#Weekend Indicator
df["is_weekend"] = df["start_date"].dt.weekday >= 5


#price per night
df["price_per_night"] = df["gross_earnings"] / df["nights"]

df["listing"] = df["listing"].astype("category")





In [68]:
#sorting values by start date
df = df.sort_values(by="start_date")

In [71]:
#final cleaned data
df_clean = df[[
    "booking_date",
    "start_date",
    "end_date",
    "listing",
    "guest",
    "nights",
    "gross_earnings",
    "price_per_night",
    "lead_time",
    "month",
    "weekday",
    "weekday_dept",
    "is_weekend"
]]
df_clean.to_csv("data/processed/airbnb_bookings_cleaned.csv", index=False)

In [70]:
df_clean.head()
df_clean.info()
df_clean.describe()
df_clean.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 581 entries, 0 to 580
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   booking_date     581 non-null    datetime64[ns]
 1   start_date       581 non-null    datetime64[ns]
 2   end_date         581 non-null    datetime64[ns]
 3   listing          581 non-null    category      
 4   guest            581 non-null    int64         
 5   nights           581 non-null    float64       
 6   gross_earnings   581 non-null    float64       
 7   price_per_night  581 non-null    float64       
 8   lead_time        581 non-null    int64         
 9   month            581 non-null    int64         
 10  weekday          581 non-null    object        
 11  weekday_dept     581 non-null    object        
 12  is_weekend       581 non-null    bool          
dtypes: bool(1), category(1), datetime64[ns](3), float64(3), int64(3), object(2)
memory usage: 56.0+

booking_date       0
start_date         0
end_date           0
listing            0
guest              0
nights             0
gross_earnings     0
price_per_night    0
lead_time          0
month              0
weekday            0
weekday_dept       0
is_weekend         0
dtype: int64